In [ ]:
# ---------------------------------------------------------
# 런타임 유형을 변경하고, 잘 적용되었는지 확인용입니다.
# ---------------------------------------------------------
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
    print('Not connected to a GPU')
else:
    print(gpu_info)

Thu Jun 11 08:39:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# <mark> 2번 : 이미지 데이터셋과 사전 훈련된 VGG16 모델을 가져와 전이 학습을 수행하세요.</mark>

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 위클리챌린지 1번에 이어, 똑같은 데이터 사용
# ---------------------------------------------------------

!curl -L "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz" -o flowers.tgz
!tar -xzf flowers.tgz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  218M  100  218M    0     0  33.8M      0  0:00:06  0:00:06 --:--:-- 38.3M


In [ ]:
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from torchvision import models

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 데이터 전처리 -> 로더 연결 -> GPU로 올리기
# ---------------------------------------------------------

transform = transforms.Compose([transforms.Resize((224, 224)),transforms.ToTensor(),])
full_dataset = ImageFolder(root='./flower_photos', transform=transform)

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader = [(inputs.to(device), labels.to(device)) for inputs, labels in train_loader]
test_loader = [(inputs.to(device), labels.to(device)) for inputs, labels in test_loader]

num_classes = 5
print(f"새로운 클래스 개수: {num_classes}")
print(f"학습 데이터 개수: {train_size}")

새로운 클래스 개수: 5
학습 데이터 개수: 2936


In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# Training from Scratch 해보기 -- 해보고 싶어서.
# ---------------------------------------------------------

base_model_Scratch = models.vgg16(weights=None)

In [ ]:
# ----------------------- 🤖 제미나이 -----------------------
# 질문:
# 앞에 ResNet50 에서는 뒤에 2개의 층을 잘랐는데, VGG16은 딱히 자르진 않네?
# resnet50의 층 구조와 vgg16의 층 구조를 비교해서 설명해줘

# 대답 :
# VGG16    |  16개(합성곱13개 + 전결합3개)   | 맥스 풀링(MaxPool2d) 후 평탄화       | 3개의 거대한 전결합층 (nn.Classifier)
# ResNet50 | 50개 (합성곱 49개 + 전결합 1개) | 글로벌 평균 풀링(AdaptiveAvgPool2d) | 단 1개의 전결합층 (nn.Linear)
#
# PyTorch에서 models.vgg16() 객체를 생성하면 크게 두 가지 덩어리로 깔끔하게 분리되어 있습니다.
# base_model.features: 이미지에서 특징을 뽑아내는 합성곱(Convolution)과 맥스풀링 층들의 모음입니다.
# base_model.classifier: 특징을 바탕으로 1000개 클래스를 분류하는 전결합층(Linear) 3개의 모음입니다.
#
# ResNet을 커스텀할 때 뒤에서 2개 층을 자른다는 의미는 보통 이렇습니다.
# 맨 마지막 fc층(1000개 분류)은 내 데이터셋 개수(예: 5개)와 맞지 않으니 잘라냅니다.
# 그 바로 앞의 avgpool층은 굳이 내가 다시 만들 필요 없이 ResNet이 쓰던 것을 그대로 써도 되지만,
# 커스텀 모델 클래스 내부에서 nn.AdaptiveAvgPool2d((1, 1))를 직접 정의해서 쓰고 싶다면
# 아예 fc와 avgpool을 둘 다 잘라내고 순수한 합성곱 블록(layer4까지)만 남기기도 합니다.
# ---------------------------------------------------------

In [ ]:
class CustomVGG16_Scratch(nn.Module):
    def __init__(self, num_classes):
        super(CustomVGG16_Scratch, self).__init__()
        self.base_model = base_model_Scratch
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

model_Scratch = CustomVGG16_Scratch(num_classes=num_classes)

model_Scratch = model_Scratch.to(device)

In [ ]:
criterion_scratch = nn.CrossEntropyLoss()
optimizer_scratch = optim.Adam(model_Scratch.parameters(), lr=0.0001)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    model_Scratch.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer_scratch.zero_grad()
        outputs = model_Scratch(inputs)
        loss = criterion_scratch(outputs, labels)
        loss.backward()
        optimizer_scratch.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 1.6040765319181525
Epoch 2, Loss: 1.540287488180658
Epoch 3, Loss: 1.4621190752672113
Epoch 4, Loss: 1.4095512408277262
Epoch 5, Loss: 1.4309183760829594
Epoch 6, Loss: 1.544099415126054
Epoch 7, Loss: 1.5071101706960928
Epoch 8, Loss: 1.4922154455081276
Epoch 9, Loss: 1.4728951804015948
Epoch 10, Loss: 1.446737043235613


In [ ]:
# ----------------------- 🤖 제미나이 -----------------------
# 학습결과
# Epoch 1, Loss: 1.6040765319181525
# Epoch 2, Loss: 1.540287488180658
# Epoch 3, Loss: 1.4621190752672113
# Epoch 4, Loss: 1.4095512408277262
# Epoch 5, Loss: 1.4309183760829594
# Epoch 6, Loss: 1.544099415126054
# Epoch 7, Loss: 1.5071101706960928
# Epoch 8, Loss: 1.4922154455081276
# Epoch 9, Loss: 1.4728951804015948
# Epoch 10, Loss: 1.446737043235613

# 질문:
# 왜 Loss가 왔다갔다해?
# ResNet을 커스텀할 때 뒤에서 2개 층을 자른다는 의미는 보통 이렇습니다.

# 대답:
# 1) 학습률(Learning Rate)이 모델의 보폭
# 현재 설정된 lr=0.0001은 사전 학습된 모델에는 부드럽게 맞지만,
# 아무것도 모르는 깡통 모델이 정답을 찾아가기에는 보폭이 조금 큰 상태일 수 있습니다.
# 2) VGG16 바닥 학습은 원래 초반에 엄청나게 방황합니다
# VGG16은 층이 깊고 파라미터가 굉장히 많은 거대한 모델입니다.
# 이미지에서 선이나 면을 구별하는 기초적인 방법조차 모르는 상태(None)이기 때문에, 학습 초반 10에포크 동안은 갈피를 잡지 못하고 이리저리 헤매며 가중치를 탐색합니다.
# 그래서 전이 학습처럼 예쁘게 일직선으로 Loss가 떨어지지 않고 지그재그를 그리게 됩니다.
# ---------------------------------------------------------

In [ ]:
model_Scratch.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model_Scratch(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

Test Accuracy: 42.51%


In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# Transfer Learning 해보기 -- 과제 조건.
# ---------------------------------------------------------

# ----------------------- 🤖 제미나이 -----------------------
# 질문:
# base_model = models.vgg16(weights=None)으로 모델을 불러오면 바닥부터 다시 학습하는걸로
# 아는데, 전이학습을 하려고 해. 그럼 models.vgg16 안에 설정을 어떻게 하면돼?

# 대답 :
# PyTorch에서 이미 ImageNet 데이터셋으로 사전 훈련된(Pre-trained) 가중치를 가져와
# 전이 학습을 진행하려면, weights 옵션에 사전 훈련된 가중치를 지정해 주어야 합니다.
# 최신 버전에서는 weights 인자에 구체적인 가중치 버전을 명시하는 것을 권장합니다.
# 가장 기본적이고 널리 쓰이는 가중치를 가져오려면 VGG16_Weights.DEFAULT를 사용하시면 됩니다.
# ---------------------------------------------------------

print(models.VGG16_Weights.DEFAULT)

VGG16_Weights.IMAGENET1K_V1


In [ ]:
base_model_TL = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

In [ ]:
class CustomVGG16_TL(nn.Module):
    def __init__(self, num_classes):
        super(CustomVGG16_TL, self).__init__()
        self.base_model_TL = base_model_TL
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1 = nn.Linear(512, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.base_model_TL.features(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return x

model_TL = CustomVGG16_TL(num_classes=num_classes)

model_TL = model_TL.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_TL.parameters(), lr=0.0001)

In [ ]:
# ----------------------- 🗣️ 내 정리  -----------------------
# 앞에서 바닥부터 학습시킨 것과 동일하게 10번 epoch 실행해서 비교해보기
# ---------------------------------------------------------

num_epochs = 10

for epoch in range(num_epochs):
    model_TL.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model_TL(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 1.2525617972664211
Epoch 2, Loss: 1.0477328481881514
Epoch 3, Loss: 1.0120634067317713
Epoch 4, Loss: 0.9981134987395742
Epoch 5, Loss: 1.0003930472809335
Epoch 6, Loss: 0.9866541345482287
Epoch 7, Loss: 0.993716097396353
Epoch 8, Loss: 0.9800964917825616
Epoch 9, Loss: 0.9708840678567472
Epoch 10, Loss: 0.9479162867950357


In [ ]:
model_TL.eval()
correct = 0
total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = model_TL(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy1 = correct / total
print(f'바닥부터 학습했을 때 Test Accuracy: {accuracy * 100:.2f}%')
print(f'models.VGG16_Weights.DEFAULT를 불러와서 전이학습했을 때 Test Accuracy: {accuracy1 * 100:.2f}%')

바닥부터 학습했을 때 Test Accuracy: 42.51%
models.VGG16_Weights.DEFAULT를 불러와서 전이학습했을 때 Test Accuracy: 92.23%


In [ ]:
# ----------------------- 🤖 제미나이 -----------------------
# 추가:
# 바닥에서부터 학습하던 model_Scratch에다가 이어서, 과적합 모니터링 구축 :
# 학습 루프 내에 실시간 Validation 검증 코드를 추가하고,
# EarlyStopping 기법을 적용하여 최적의 타이밍에 학습을 종료하는 자동화 시스템을 보완.
# ---------------------------------------------------------


import copy

# 💡 Early Stopping 기준점을 0으로 낮춰서, 초반에 무조건 가중치를 업데이트하며 지나가게 만듭니다.
best_acc = 0.0
best_model_wts = copy.deepcopy(model_Scratch.state_dict())

print(f"▶️ 10번 셀 가중치를 그대로 이어받아 11번째 에포크부터 추가 학습을 시작합니다.")

# 💡 바닥 학습은 성능이 더디게 오르므로 참을성(patience)을 5나 10으로 넉넉하게 늘려줍니다.
patience = 5
patience_counter = 0
extra_epochs = 10     # 추가로 10번 더 돌려봅니다 (Epoch 11 ~ 20)

print(f"=== 11번째 에포크부터 이어서 학습 시작 ===")

for epoch in range(extra_epochs):
    model_Scratch.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer_scratch.zero_grad()
        outputs = model_Scratch(inputs)
        loss = criterion_scratch(outputs, labels)
        loss.backward()
        optimizer_scratch.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    model_Scratch.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model_Scratch(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total

    # 출력할 때 현재 몇 번째 에포크인지 명확하게 인지하도록 11부터 찍히게 합니다.
    print(f'Epoch [{epoch + 11}] | Train Loss: {train_loss:.4f} | Val Accuracy: {val_acc*100:.2f}%')

    if val_acc > best_acc:
        best_acc = val_acc
        best_model_wts = copy.deepcopy(model_Scratch.state_dict())
        patience_counter = 0
        print(f"  └ ⭐ 최고 정확도 경신! 가중치 백업 ({best_acc*100:.2f}%)")
    else:
        patience_counter += 1
        print(f"  └ ⚠️ 성능 향상 없음 (조기 종료 카운트: {patience_counter}/{patience})")

    if patience_counter >= patience:
        print(f"\n🛑 성능 향상이 지체되어 {patience}에포크 만에 조기 종료합니다.")
        break

model_Scratch.load_state_dict(best_model_wts)

# 앞에서 10번까지 학습했던 모델의 결과
# Epoch 10, Loss: 1.446737043235613
# Test Accuracy: 42.51%
print(f'\n🏆 학습 종료! 최종 최적의 Test Accuracy: {best_acc * 100:.2f}%')

▶️ 10번 셀 가중치를 그대로 이어받아 11번째 에포크부터 추가 학습을 시작합니다.
=== 11번째 에포크부터 이어서 학습 시작 ===
Epoch [11] | Train Loss: 1.4314 | Val Accuracy: 47.68%
  └ ⭐ 최고 정확도 경신! 가중치 백업 (47.68%)
Epoch [12] | Train Loss: 1.4261 | Val Accuracy: 45.37%
  └ ⚠️ 성능 향상 없음 (조기 종료 카운트: 1/5)
Epoch [13] | Train Loss: 1.4132 | Val Accuracy: 47.28%
  └ ⚠️ 성능 향상 없음 (조기 종료 카운트: 2/5)
Epoch [14] | Train Loss: 1.4010 | Val Accuracy: 50.82%
  └ ⭐ 최고 정확도 경신! 가중치 백업 (50.82%)
Epoch [15] | Train Loss: 1.3649 | Val Accuracy: 51.09%
  └ ⭐ 최고 정확도 경신! 가중치 백업 (51.09%)
Epoch [16] | Train Loss: 1.3476 | Val Accuracy: 56.68%
  └ ⭐ 최고 정확도 경신! 가중치 백업 (56.68%)
Epoch [17] | Train Loss: 1.3147 | Val Accuracy: 55.72%
  └ ⚠️ 성능 향상 없음 (조기 종료 카운트: 1/5)
Epoch [18] | Train Loss: 1.3368 | Val Accuracy: 60.90%
  └ ⭐ 최고 정확도 경신! 가중치 백업 (60.90%)
Epoch [19] | Train Loss: 1.3031 | Val Accuracy: 61.44%
  └ ⭐ 최고 정확도 경신! 가중치 백업 (61.44%)
Epoch [20] | Train Loss: 1.3022 | Val Accuracy: 59.13%
  └ ⚠️ 성능 향상 없음 (조기 종료 카운트: 1/5)

🏆 학습 종료! 최종 최적의 Test Accuracy: 61.44%


# <font color='red'>1. 배운 점</font>


<font color='green'>1.   weights 인자 하나로 갈리는 두 가지 학습 패러다임
- `models.vgg16(weights=None)`: 구조(설계도)만 가져오고 가중치는 무작위 초기화되는 Training from Scratch.
- `models.vgg16(weights=models.VGG16_Weights.DEFAULT)`: ImageNet 사전 학습 가중치를 불러오는 Transfer Learning.   
  *  DEFAULT는 임의의 정수가 아니라 내부적으로 IMAGENET1K_V1이라는 검증된 가중치 파일을 가리키는 이름표(Enum).
2.  ResNet50 vs VGG16 구조 비교
- ResNet50은 합성곱 49개가 하나의 흐름으로 이어지다 avgpool + fc 1개로 끝나서 "뒤에서 층을 잘라내는" 방식을 쓰는 반면, VGG16은 features(합성곱 13개)와 classifier(전결합 3개)가 변수로 깔끔하게 분리되어 있어 `base_model.features`만 이름으로 쏙 꺼내 쓰는 방식을 사용.
3.   변수 참조와 가중치 생존의 원리
- 모델 객체를 새로 선언(`model = CustomVGG16()`)하는 순간 이전 학습 가중치는 완전히 사라지고 무작위 초기화된 새 모델이 탄생함을 확인. Scratch 모델과 전이학습 모델을 동시에 보존하려면 `_Scratch`, `_TL` 접미사로 변수명을 분리해 독립된 객체로 관리해야 함을 학습.
- 옵티마이저는 선언 시점의 `model.parameters()` 주소를 붙잡고 있으므로, 모델을 새로 만들면 옵티마이저도 반드시 다시 연결해야 하며, 반대로 가중치를 이어서 학습하려면 모델·옵티마이저 선언 셀을 절대 재실행하면 안 된다는 점을 체득.


# <font color='red'>2. 어려웠던 점</font>


<font color='green'>1.   가중치가 얼어붙은 모델 — 옵티마이저와 모델의 연결 불일치
- 학습 루프에서 Loss와 Accuracy가 에포크마다 소수점 끝자리까지 완벽하게 동일한 값으로 고정되는 현상을 마주.
- 원인은 옵티마이저 선언 시 변수명(optimizer)과 학습 루프 내 변수명(optimizer_scratch)이 달라, 가중치 업데이트 명령이 엉뚱한 곳으로 가면서 모델이 단 한 번도 학습되지 않았던 것.
- 가중치가 안 변하니 Early Stopping이 "성능 향상 없음"으로 판단해 즉시 조기 종료해버리는 연쇄 반응까지 경험.
2.   노트북 셀 실행 순서와 변수 오염과의 사투
- 전이학습을 위해 base_model과 model을 덮어쓰면서 앞서 학습한 Scratch 모델이 메모리에서 사라지는 오염을 겪었고, 학습 루프의 변수명을 안 바꿔 완성해 둔 전이학습 모델을 다시 바닥부터 덮어쓸 뻔한 위기도 경험. 이외에도 출력 시 accuracy/accuracy1 변수 혼용으로 두 모델 정확도가 똑같이 찍히는 버그, `model = model.to(device)` 오타로 Scratch 모델이 GPU에 못 올라가는 버그, forward 내부의 `self.base_model` 참조 오타 등 변수명 관련 버그를 연달아 디버깅.
3.   진짜 고장과 정상 동작의 구분
- Loss가 1.60 → 1.41까지 내려갔다 다시 1.54로 튀는 지그재그를 보고 모델이 터진 것인지 의심.
- 실제로는 (1) shuffle로 매 에포크 배치 조합의 난이도가 달라지고, (2) 깡통 모델에게 lr=0.0001이 다소 커서 정답 골짜기를 지나치는 오버슈팅이 발생하며, (3) 깊은 VGG16의 바닥 학습은 원래 초반에 방황한다는 정상적인 탐색 과정임을 이해.
- "소수점까지 완전히 똑같이 굳은 Loss(고장)"와 "출렁이는 Loss(정상 탐색)"를 구분하는 진단 기준을 획득.



# <font color='red'>3. 앞으로의 보완점 -- 🤖 클로드 제안</font>


<font color='green'>1.   출력층의 Softmax 중복 제거
- `nn.CrossEntropyLoss`는 내부에 LogSoftmax + NLLLoss가 이미 결합되어 있으므로, forward 마지막의 `self.softmax(x)`는 소프트맥스 이중 적용이 되어 그래디언트를 약화시킴. 출력층에서 Softmax를 제거하고 로짓을 그대로 손실 함수에 전달하도록 수정.
2.   torch.save 기반 체크포인트 습관화
- 이번처럼 변수를 덮어써서 학습된 가중치를 잃는 사고를 막기 위해, 의미 있는 학습이 끝날 때마다 `torch.save(model.state_dict())`로 파일 백업하는 습관을 보완.
3.   Validation / Test 데이터 분리 및 입력 정규화
- 현재 Early Stopping 검증에 test_loader를 그대로 사용해 테스트 정보가 학습 종료 결정에 새어 들어가는 구조이므로 train/validation/test 3분할로 개선. 또한 사전 학습 가중치는 ImageNet 통계로 정규화된 입력을 전제로 하므로 `transforms.Normalize` 추가 필요.


# <font color='red'>4. 수행 과정 종합 평가</font>


<font color='green'>1.   서론
- 본 과제는 꽃 이미지 데이터셋 'flower_photos, 5개 클래스'을 대상으로, 1) 동일한 VGG16 아키텍처를 무작위 초기화 상태의 바닥 학습 'Scratch' 과 2) ImageNet 사전 학습 가중치 기반 전이 학습 'Transfer Learning' 두 방식으로 각각 훈련시켜 성능을 정면 비교하는 것을 목표로 진행되었습니다.
- features 뒤에 글로벌 평균 풀링과 2단 전결합층(512→256→5)을 직접 설계한 커스텀 구조를 양쪽에 동일하게 적용하고, Google Colab GPU 환경에서 모델과 데이터를 `.to(device)`로 동기화하여 수행했습니다.

<font color='green'>2.   본론


학습은 총 세 단계의 연속적인 실행 및 관찰 프로세스로 전개되었습니다.

- 1차 바닥 학습(Scratch):
`weights=None`의 백지 VGG16을 10 에포크 학습한 결과, Loss가 1.60에서 1.41까지 내려갔다 다시 튀어 오르는 지그재그 패턴을 보이며 최종 테스트 정확도 42.51%에 머물렀습니다. 이 진동이 미니배치 셔플·오버슈팅·깊은 모델의 초기 탐색에서 비롯된 정상 현상임을 분석했습니다.
- 2차 전이 학습(Transfer Learning):
  *   두 모델을 동시에 보존하기 위해 변수명을 `_Scratch`와 `_TL`로 분리하는 리팩토링을 거쳐, 동일 조건(10 에포크, lr=0.0001)에서 사전 학습 가중치만 갈아 끼운 결과 테스트 정확도 88.01%를 기록했습니다. 오직 초기 가중치의 차이만으로 정확도가 2배 이상 갈리는 것을 정량적으로 확인했습니다.
- 3차 검증 자동화 및 누적 학습:
  *   지난 과제의 보완점이었던 과적합 모니터링을 실제로 구축했습니다.
  - 이 과정에서 옵티마이저-모델 연결 불일치로 가중치가 전혀 업데이트되지 않아 Loss가 동결되고 Early Stopping이 오작동하는 사고를 디버깅했으며,
  - 최종적으로 Scratch 모델의 가중치를 보존한 채 11~20 에포크를 누적 학습하면서 매 에포크 검증 정확도를 측정하고, 최고 기록 경신 시 `copy.deepcopy`로 가중치를 백업, patience(5회) 초과 시 자동 종료되는 시스템을 완성했습니다.

<font color='green'>3.   결론
- 최종적으로 Early Stopping이 백업해 둔 최적 가중치를 `load_state_dict`로 복원하여, 바닥 학습 모델의 정확도를 42.51%에서 61.44%까지 끌어올렸습니다.
- 동일한 구조·데이터·하이퍼파라미터에서 Scratch(42.51%) 대 Transfer Learning(88.01%)이라는 압도적 격차를 직접 비교 실험으로 체감하며, 사전 학습된 지식을 빌려오는 전이 학습의 위력을 확인했습니다.
- 또한 수많은 변수 오염 사고와 디버깅을 통해, 파이토치에서 모델 객체·옵티마이저·가중치가 메모리에서 어떻게 연결되고 끊어지는지를 코드가 아닌 경험으로 이해하게 되었으며, 검증 성능 기준으로 "언제 멈출지"를 모델이 스스로 판단하는 자동화 파이프라인을 구축했습니다.

🤖 제미나이 대화 원본 : https://gemini.google.com/share/48e0d989ab9c


수행시간 : **1시반 30분 소요**